# Motion Planning

Source orientation: printed pages 353-402, PDF pages 371-420. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

How does a robot find a feasible path through a high-dimensional space with obstacles?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: C-space obstacle, complete planner, grid, RRT, PRM, potential field, smoothing. The important conversions for this notebook are:

- Collision checking defines the free subset of configuration space.
- Search trades resolution, optimality, and speed.
- Sampling planners replace exhaustive grids with exploration bias.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Route Through the Notebook

1. Establish the objects and conventions needed for motion planning.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: configuration-space obstacles and graph search.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-10-motion-planning/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| configuration-space obstacles and graph search | grid planner, obstacle field, and sampling intuition | `artifacts/chapter-10-motion-planning/figures/motion-planning-lab.png` | how collision constraints turn geometry into search |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-10-motion-planning/figures/motion-planning-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Reading The Planning Lab

Motion planning starts by changing the question. Instead of asking whether the robot's body can move through workspace, we ask whether a point can move through configuration space. For a mobile robot that point might be a pose; for a manipulator it might be a vector of joint angles. A collision checker turns each candidate configuration into a yes/no statement, so the planner sees the world as a set of free configurations separated by forbidden regions. This translation is powerful because it lets very different machines share the same search vocabulary, but it is also the source of the chapter's difficulty: a simple table, wall, or link collision in workspace can carve a twisted obstacle through a high-dimensional C-space.

The grid panel should be read as a small model of that translation. Black cells are configurations rejected by the collision model, white cells are admissible samples, and the blue curve is a discrete path through adjacent free cells. A grid planner is resolution complete rather than magically complete: if a free corridor is wider than the grid can resolve, the graph search can find it; if the corridor disappears between samples, the planner can confidently solve the wrong problem. This is why the visual is paired with a path-node count instead of only a pretty curve. The count gives a machine-checkable trace that the search actually reached the goal.

Sampling planners such as PRM and RRT make a different bargain. They avoid enumerating every grid cell, so they can explore spaces whose dimension would make a full grid unusable. In return, they reason probabilistically: more samples improve coverage, nearest-neighbor choices bias the tree or roadmap, and narrow passages remain hard because random samples rarely land where the geometry is most constrained. Potential fields make still another bargain by turning the goal into attraction and obstacles into repulsion. Their paths can look smooth and purposeful, yet the field can contain local minima that trap the robot before it reaches the goal.

Smoothing belongs after feasibility, not before it. A jagged graph path is evidence that the planner found a collision-free route in the discrete model; smoothing tries to shorten or regularize that route while preserving clearance. The invariant to protect is therefore not visual smoothness but membership in free space. Any shortcut, spline, or time scaling must be checked against the same collision and dynamic limits that made the original path meaningful.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For motion planning, the relevant failure modes are not side details; they are part of the concept. Singularities, chart boundaries, rank loss, time-scaling limits, contact mode changes, and wheel constraints are all examples of geometry asserting itself. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Solve a grid planning problem and compare the resulting path with a potential-field intuition.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, ask whether a rank should change, whether a path should lengthen, whether an eigenvalue should stay positive, whether a cone should widen, or whether a coordinate chart should approach a singular value. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- A workspace obstacle becomes a complicated C-space obstacle.
- Potential fields can create false local traps.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Planning is geometry plus search policy.
- Resolution and sampling choices are part of the model.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 10,
  "slug": "chapter-10-motion-planning",
  "title": "Motion Planning",
  "notebook": "10-motion-planning.ipynb",
  "printed_start": 353,
  "printed_end": 402,
  "pdf_start": 371,
  "pdf_end": 420,
  "part_slug": "part-03-dynamics-trajectories-and-planning",
  "part_title": "Dynamics, Trajectories, and Planning",
  "theme": "planning",
  "visual_focus": "configuration-space obstacles and graph search",
  "visual_kind": "grid planner, obstacle field, and sampling intuition",
  "artifact_stem": "motion-planning",
  "inspection_target": "how collision constraints turn geometry into search",
  "question": "How does a robot find a feasible path through a high-dimensional space with obstacles?",
  "terms": [
    "C-space obstacle",
    "complete planner",
    "grid",
    "RRT",
    "PRM",
    "potential field",
    "smoothing"
  ],
  "translation": [
    "Collision checking defines the free subset of configuration space.",
    "Search trades resolution, optimality, and speed.",
    "Sampling planners replace exhaustive grids with exploration bias."
  ],
  "lab": "Solve a grid planning problem and compare the resulting path with a potential-field intuition.",
  "pitfalls": [
    "A workspace obstacle becomes a complicated C-space obstacle.",
    "Potential fields can create false local traps."
  ],
  "takeaways": [
    "Planning is geometry plus search policy.",
    "Resolution and sampling choices are part of the model."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

A planner is only as trustworthy as the representations underneath its collision and motion queries. The worked example below checks three small ingredients that planning code often depends on: a planar arm Jacobian for local motion directions, a rotation exponential/logarithm round trip for pose coordinates, and the twist/wrench power pairing under a frame transform. None of these is a full planner by itself. They are compact audits of the coordinate discipline that lets a path in configuration space correspond to a physically meaningful robot motion.

In [ ]:
from utils.kinematics import planar_arm_points, planar_jacobian, manipulability_measure
from utils.lie import adjoint, se3_exp, se3_log, so3_exp, so3_log, transform_from, wrench_power

lengths = np.array([1.0, 0.75, 0.45])
theta = np.array([0.45, -0.8, 0.6])
points = planar_arm_points(lengths, theta)
J = planar_jacobian(lengths, theta)
R = so3_exp(np.array([0.2, -0.1, 0.35]))
round_trip = np.linalg.norm(so3_log(R) - np.array([0.2, -0.1, 0.35]))
T = transform_from(R, np.array([0.25, -0.1, 0.4]))
twist = np.array([0.1, -0.2, 0.3, 0.4, 0.2, -0.1])
wrench = np.array([0.3, 0.1, -0.2, 1.0, -0.4, 0.2])
power_before = wrench_power(twist, wrench)
power_after = wrench_power(adjoint(T) @ twist, np.linalg.inv(adjoint(T)).T @ wrench)
worked_example = {
    "endpoint": points[-1].round(4).tolist(),
    "jacobian_rank": int(np.linalg.matrix_rank(J)),
    "manipulability": float(manipulability_measure(J)),
    "rotation_log_round_trip": float(round_trip),
    "power_pairing_error": float(abs(power_before - power_after)),
}
worked_example

## Applied Lab

The applied lab now mirrors the visual grid planner. It rebuilds the same obstacle field, asks Dijkstra search for a free path, then closes a vertical barrier to confirm that the planner reports no route when the C-space graph is disconnected. The midpoint time-scaling values stay in the summary because feasible geometric paths still need a timing law before becoming executable trajectories.

In [ ]:
from utils.control import pd_response
from utils.dynamics import two_link_mass_matrix
from utils.grasping import friction_cone, grasp_matrix
from utils.mobile import mecanum_wheel_matrix, unicycle_rollout
from utils.planning import cubic_time_scaling, dijkstra_grid, quintic_time_scaling

theme = CHAPTER["theme"]
if theme in {"configuration", "kinematics"}:
    sweep = np.linspace(-np.pi, np.pi, 90)
    values = [manipulability_measure(planar_jacobian(np.array([1.0, 0.8]), np.array([0.25, q]))) for q in sweep]
    lab_summary = {"theme": theme, "max_manipulability": float(np.max(values)), "min_manipulability": float(np.min(values))}
elif theme in {"rigid", "appendix"}:
    angles = np.linspace(0.0, 0.95 * np.pi, 60)
    residuals = [np.linalg.norm(so3_log(so3_exp(np.array([0.0, 0.0, a]))) - np.array([0.0, 0.0, a])) for a in angles]
    lab_summary = {"theme": theme, "max_rotation_residual": float(np.max(residuals))}
elif theme == "dynamics":
    eigs = np.array([np.linalg.eigvalsh(two_link_mass_matrix(q)) for q in np.linspace(-np.pi, np.pi, 80)])
    lab_summary = {"theme": theme, "minimum_mass_eigenvalue": float(eigs.min())}
elif theme == "planning":
    cost = np.ones((42, 42))
    cost[12:30, 18:24] = np.inf
    cost[6:12, 5:31] = np.inf
    start, goal = (36, 4), (5, 36)
    path = dijkstra_grid(cost, start, goal)
    blocked = cost.copy()
    blocked[:, 18:24] = np.inf
    blocked_path = dijkstra_grid(blocked, start, goal)
    manhattan_edges = abs(start[0] - goal[0]) + abs(start[1] - goal[1])
    lab_summary = {
        "theme": theme,
        "path_nodes": int(len(path)),
        "path_edges": int(max(len(path) - 1, 0)),
        "manhattan_lower_bound": int(manhattan_edges),
        "barrier_closes_route": bool(len(blocked_path) == 0),
        "cubic_midpoint": float(cubic_time_scaling(np.array([0.5]), 1.0)[0]),
        "quintic_midpoint": float(quintic_time_scaling(np.array([0.5]), 1.0)[0]),
    }
elif theme == "contact":
    cone = friction_cone(np.array([0.0, 1.0]), 0.5)
    G = grasp_matrix(np.array([[1.0, 0.0], [-0.5, 0.86], [-0.5, -0.86]]), np.array([[-1.0, 0.0], [0.5, -0.86], [0.5, 0.86]]))
    lab_summary = {"theme": theme, "cone_samples": int(len(cone)), "grasp_rank": int(np.linalg.matrix_rank(G))}
elif theme == "mobile":
    controls = np.c_[np.ones(80) * 0.5, np.ones(80) * 0.3]
    path = unicycle_rollout(controls)
    lab_summary = {"theme": theme, "wheel_map_rank": int(np.linalg.matrix_rank(mecanum_wheel_matrix())), "final_pose": path[-1].round(4).tolist()}
else:
    lab_summary = {"theme": theme}

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity